In [ ]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd()
OUT_DIR = ROOT / "data" / "output"
if not OUT_DIR.exists():
    OUT_DIR = ROOT.parent / "data" / "output"

parquet_path = OUT_DIR / "forecast.parquet"
if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

raw_df = pl.read_parquet(parquet_path)

In [136]:
raw_df.head()

unique_id,ds,value,valuehat,y,yhat,train_start,train_end,test_start,test_end,forecast_start,forecast_end,sku_desc,store_name,seccion,period_type,driver_effect,driver_effect_value,yhat_seccion,valuehat_seccion,yhat_tienda,valuehat_tienda,modelo_seleccionado
str,date,f64,f64,f64,f64,date,date,date,date,date,date,str,str,str,str,f64,f64,f32,f32,f32,f32,str
"""1""",2024-05-26,2134653.8,2.0427e6,17188.0,16525.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-27,1.9304e6,2.0117e6,16029.0,16911.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-28,2.0906e6,2.2787e6,17250.0,18786.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-29,1.8514e6,1.9099e6,16003.0,16741.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-30,1.7116e6,1.8177e6,14036.0,15013.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null


In [137]:
df = raw_df.filter((pl.col("y") != 0)).with_columns(
    (pl.col("value") - pl.col("valuehat")).abs().alias("abs_err")
)["unique_id", "ds", "value", "valuehat", "abs_err", "period_type"]
print(df.head())

shape: (5, 6)
┌───────────┬────────────┬───────────┬──────────┬───────────┬─────────────┐
│ unique_id ┆ ds         ┆ value     ┆ valuehat ┆ abs_err   ┆ period_type │
│ ---       ┆ ---        ┆ ---       ┆ ---      ┆ ---       ┆ ---         │
│ str       ┆ date       ┆ f64       ┆ f64      ┆ f64       ┆ str         │
╞═══════════╪════════════╪═══════════╪══════════╪═══════════╪═════════════╡
│ 1         ┆ 2024-05-26 ┆ 2134653.8 ┆ 2.0427e6 ┆ 91914.67  ┆ in_sample   │
│ 1         ┆ 2024-05-27 ┆ 1.9304e6  ┆ 2.0117e6 ┆ 81269.57  ┆ in_sample   │
│ 1         ┆ 2024-05-28 ┆ 2.0906e6  ┆ 2.2787e6 ┆ 188084.34 ┆ in_sample   │
│ 1         ┆ 2024-05-29 ┆ 1.8514e6  ┆ 1.9099e6 ┆ 58553.66  ┆ in_sample   │
│ 1         ┆ 2024-05-30 ┆ 1.7116e6  ┆ 1.8177e6 ┆ 106086.21 ┆ in_sample   │
└───────────┴────────────┴───────────┴──────────┴───────────┴─────────────┘


In [148]:
sku_df = df.filter(
    (pl.col("unique_id") == "1||T:00063||S:127360")
    # & (pl.col("period_type") == "in_sample")
)
print(sku_df.head())

shape: (5, 6)
┌──────────────────────┬────────────┬─────────┬──────────┬─────────┬─────────────┐
│ unique_id            ┆ ds         ┆ value   ┆ valuehat ┆ abs_err ┆ period_type │
│ ---                  ┆ ---        ┆ ---     ┆ ---      ┆ ---     ┆ ---         │
│ str                  ┆ date       ┆ f64     ┆ f64      ┆ f64     ┆ str         │
╞══════════════════════╪════════════╪═════════╪══════════╪═════════╪═════════════╡
│ 1||T:00063||S:127360 ┆ 2024-05-26 ┆ 980.1   ┆ 980.1    ┆ 0.0     ┆ in_sample   │
│ 1||T:00063||S:127360 ┆ 2024-05-27 ┆ 980.1   ┆ 1055.28  ┆ 75.18   ┆ in_sample   │
│ 1||T:00063||S:127360 ┆ 2024-05-28 ┆ 980.1   ┆ 1284.41  ┆ 304.31  ┆ in_sample   │
│ 1||T:00063||S:127360 ┆ 2024-05-29 ┆ 495.0   ┆ 959.98   ┆ 464.98  ┆ in_sample   │
│ 1||T:00063||S:127360 ┆ 2024-05-30 ┆ 1678.05 ┆ 993.2    ┆ 684.85  ┆ in_sample   │
└──────────────────────┴────────────┴─────────┴──────────┴─────────┴─────────────┘


In [149]:
sku_df.count()

unique_id,ds,value,valuehat,abs_err,period_type
u32,u32,u32,u32,u32,u32
688,688,688,688,688,688


In [140]:
print(sku_df["value", "valuehat", "abs_err"].sum())
print(192969.01 / 632903.05)

shape: (1, 3)
┌───────────┬───────────┬───────────┐
│ value     ┆ valuehat  ┆ abs_err   │
│ ---       ┆ ---       ┆ ---       │
│ f64       ┆ f64       ┆ f64       │
╞═══════════╪═══════════╪═══════════╡
│ 632903.05 ┆ 564131.48 ┆ 192969.01 │
└───────────┴───────────┴───────────┘
0.30489505462171496


In [141]:
wMAPE_sku = sku_df["abs_err"].sum() / sku_df["value"].sum()
wMAPE_sku

0.3048950546217149

In [142]:
sto_df = df.filter(
    (pl.col("unique_id").str.contains("||T:00063", literal=True))
    & (pl.col("period_type") == "in_sample")
)
print(sto_df.head())

shape: (5, 6)
┌────────────┬────────────┬───────────┬───────────┬──────────┬─────────────┐
│ unique_id  ┆ ds         ┆ value     ┆ valuehat  ┆ abs_err  ┆ period_type │
│ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---      ┆ ---         │
│ str        ┆ date       ┆ f64       ┆ f64       ┆ f64      ┆ str         │
╞════════════╪════════════╪═══════════╪═══════════╪══════════╪═════════════╡
│ 1||T:00063 ┆ 2024-05-26 ┆ 91737.5   ┆ 101908.42 ┆ 10170.92 ┆ in_sample   │
│ 1||T:00063 ┆ 2024-05-27 ┆ 82882.76  ┆ 110621.91 ┆ 27739.15 ┆ in_sample   │
│ 1||T:00063 ┆ 2024-05-28 ┆ 118449.63 ┆ 138719.72 ┆ 20270.09 ┆ in_sample   │
│ 1||T:00063 ┆ 2024-05-29 ┆ 91743.21  ┆ 111616.04 ┆ 19872.83 ┆ in_sample   │
│ 1||T:00063 ┆ 2024-05-30 ┆ 105658.48 ┆ 108942.99 ┆ 3284.51  ┆ in_sample   │
└────────────┴────────────┴───────────┴───────────┴──────────┴─────────────┘


In [143]:
wMAPE_sto = sto_df["abs_err"].sum() / sto_df["value"].sum()
wMAPE_sto

0.4082859992564659

In [144]:
sec_df = df.filter(
    (pl.col("unique_id").str.starts_with("1||"))
    & (pl.col("period_type") == "in_sample")
)
print(sec_df.head())

shape: (5, 6)
┌────────────┬────────────┬───────────┬───────────┬──────────┬─────────────┐
│ unique_id  ┆ ds         ┆ value     ┆ valuehat  ┆ abs_err  ┆ period_type │
│ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---      ┆ ---         │
│ str        ┆ date       ┆ f64       ┆ f64       ┆ f64      ┆ str         │
╞════════════╪════════════╪═══════════╪═══════════╪══════════╪═════════════╡
│ 1||T:00063 ┆ 2024-05-26 ┆ 91737.5   ┆ 101908.42 ┆ 10170.92 ┆ in_sample   │
│ 1||T:00063 ┆ 2024-05-27 ┆ 82882.76  ┆ 110621.91 ┆ 27739.15 ┆ in_sample   │
│ 1||T:00063 ┆ 2024-05-28 ┆ 118449.63 ┆ 138719.72 ┆ 20270.09 ┆ in_sample   │
│ 1||T:00063 ┆ 2024-05-29 ┆ 91743.21  ┆ 111616.04 ┆ 19872.83 ┆ in_sample   │
│ 1||T:00063 ┆ 2024-05-30 ┆ 105658.48 ┆ 108942.99 ┆ 3284.51  ┆ in_sample   │
└────────────┴────────────┴───────────┴───────────┴──────────┴─────────────┘


In [145]:
wMAPE_sec = sec_df["abs_err"].sum() / sec_df["value"].sum()
wMAPE_sec

0.40686699406868976